# Regras de Associação e Market Basket Analysis

Projeto de **mineração de dados** usando o algoritmo Apriori para encontrar combinações frequentes de produtos e gerar regras de associação.

## Conceitos explorados
- transações e cestas de compras;
- itemsets frequentes;
- suporte;
- confiança;
- lift;
- filtragem de regras;
- comparação entre mercados (França e Alemanha).

**Tecnologias:** Python, Pandas, NumPy, Matplotlib e MLxtend.

## 1. Instalação e bibliotecas

Em um ambiente local, instale as dependências com `pip install -r requirements.txt`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

## 2. Exemplo simples

Primeiro, usamos pequenas cestas artificiais para visualizar o funcionamento do Apriori.

In [ ]:
dataset = [
    ["Leite", "Cebola", "Batata", "Feijão", "Ovos", "Iogurte"],
    ["Arroz", "Cebola", "Batata", "Feijão", "Ovos", "Iogurte"],
    ["Leite", "Maçã", "Feijão", "Ovos"],
    ["Leite", "Milho", "Feijão", "Iogurte"],
    ["Milho", "Cebola", "Feijão", "Sorvete", "Ovos"],
]

te = TransactionEncoder()
te_array = te.fit(dataset).transform(dataset)
df_demo = pd.DataFrame(te_array, columns=te.columns_)

itemsets = apriori(df_demo, min_support=0.6, use_colnames=True)
rules_demo = association_rules(itemsets, metric="confidence", min_threshold=0.7)

print(itemsets)
display(rules_demo)

## 3. Online Retail — preparação

A análise principal utiliza a base **Online Retail**, disponibilizada pela UCI. O arquivo deve estar em `data/online-retail/Online Retail.xlsx`.

O notebook original utiliza essa base para analisar compras na França e na Alemanha.

In [ ]:
from pathlib import Path

DATA_PATH = Path("data/online-retail/Online Retail.xlsx")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Adicione Online Retail.xlsx em data/online-retail/. "
        "Consulte o README para a origem da base."
    )

df = pd.read_excel(DATA_PATH)
print(df.shape)
display(df.head())

In [ ]:
df["Description"] = df["Description"].astype("string").str.strip()
df = df.dropna(subset=["InvoiceNo"]).copy()
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.contains("C", na=False)]

print("Após limpeza:", df.shape)

## 4. Função para construir uma matriz de cestas

In [ ]:
def build_basket(dataframe, country):
    basket = (
        dataframe[dataframe["Country"] == country]
        .groupby(["InvoiceNo", "Description"])["Quantity"]
        .sum()
        .unstack()
        .fillna(0)
    )

    def encode_units(value):
        return 1 if value > 0 else 0

    basket_sets = basket.map(encode_units)
    if "POSTAGE" in basket_sets.columns:
        basket_sets = basket_sets.drop(columns="POSTAGE")
    return basket, basket_sets

## 5. França

In [ ]:
basket_fr, basket_fr_sets = build_basket(df, "France")
itemsets_fr = apriori(basket_fr_sets, min_support=0.07, use_colnames=True)
rules_fr = association_rules(itemsets_fr, metric="lift", min_threshold=1)

filtered_fr = rules_fr[(rules_fr["lift"] >= 6) & (rules_fr["confidence"] >= 0.8)]
print("Itemsets frequentes:", len(itemsets_fr))
print("Regras geradas:", len(rules_fr))
display(filtered_fr.sort_values(["lift", "confidence"], ascending=False).head(20))

## 6. Alemanha

O exemplo original utiliza parâmetros diferentes para mostrar que o suporte mínimo e a confiança podem variar conforme a base analisada.

In [ ]:
basket_de, basket_de_sets = build_basket(df, "Germany")
itemsets_de = apriori(basket_de_sets, min_support=0.05, use_colnames=True)
rules_de = association_rules(itemsets_de, metric="lift", min_threshold=1)

filtered_de = rules_de[(rules_de["lift"] >= 4) & (rules_de["confidence"] >= 0.5)]
print("Itemsets frequentes:", len(itemsets_de))
print("Regras geradas:", len(rules_de))
display(filtered_de.sort_values(["lift", "confidence"], ascending=False).head(20))

## Interpretação das métricas

- **Suporte:** frequência relativa da combinação na base.
- **Confiança:** frequência com que o consequente aparece quando o antecedente ocorre.
- **Lift:** compara a ocorrência conjunta com o que seria esperado pela ocorrência independente dos itens. Valores acima de 1 indicam associação positiva segundo essa métrica; isso não implica causalidade.

## Conclusão

O projeto mostra como transformar transações em uma matriz binária, gerar itemsets frequentes com Apriori e filtrar regras de associação por métricas de interesse.